# Step 6: Sequence-to-Sequence (Seq2Seq)

Single-layer RNNs and LSTMs process one sequence — they classify tokens or predict the next one.
Many real tasks map one sequence to a **different** sequence of a **different length**:
translation, summarization, parsing, speech recognition.

**Seq2seq** (Sutskever, Vinyals & Le, 2014) uses **two** LSTMs:
- **Encoder**: reads the input sequence, compresses it into a fixed-size **context vector**
- **Decoder**: initialized with the context vector, generates the output sequence token by token

**Task**: reverse a sequence of numbers. Simple enough to train quickly;
complex enough to show the architecture's strengths and its critical weakness.

**What you'll see:**
1. Encoder-decoder architecture with a context vector bottleneck
2. Teacher forcing during training vs. autoregressive decoding at inference
3. **The bottleneck problem**: performance collapses as sequence length increases
4. How the bottleneck is visible in the context vector's information content

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)
plt.rcParams['figure.dpi'] = 100

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## Task: Sequence Reversal

Input:  `[3, 7, 2, 5, 1]`  
Output: `[1, 5, 2, 7, 3]`

This requires remembering all input elements, even the first one, when generating the last output.
A perfect stress test for whether information survives the fixed-size bottleneck.

In [ ]:
# Vocabulary: digits 0-9, plus special tokens
PAD, SOS, EOS = 10, 11, 12
VOCAB_SIZE = 13

def make_batch(batch_size, seq_len):
    """Generate (input, target) pairs for sequence reversal."""
    seqs = np.random.randint(0, 10, size=(batch_size, seq_len))
    src = torch.tensor(seqs, dtype=torch.long)                      # (B, seq_len)
    tgt_in  = torch.cat([torch.full((batch_size,1), SOS), 
                          torch.tensor(seqs[:, ::-1].copy())], dim=1)  # SOS + reversed
    tgt_out = torch.cat([torch.tensor(seqs[:, ::-1].copy()),
                          torch.full((batch_size,1), EOS)], dim=1)  # reversed + EOS
    return src, tgt_in, tgt_out

# Quick sanity check
src, tgt_in, tgt_out = make_batch(2, 5)
print("Source:    ", src[0].tolist())
print("Target in: ", tgt_in[0].tolist(), " (SOS=11 prepended)")
print("Target out:", tgt_out[0].tolist(), " (EOS=12 appended)")

## The Architecture

```
Encoder:
  for each x_t in input:
    h_t, c_t = LSTM(embed(x_t), h_{t-1}, c_{t-1})
  context = (h_T, c_T)   ← the ONLY information passed to the decoder

Decoder (teacher forcing during training):
  h, c = context
  for each y_t in target[:-1]:
    h, c = LSTM(embed(y_t), h, c)
    logits_t = fc(h)

Decoder (autoregressive at inference):
  h, c = context; x = SOS
  while x != EOS:
    h, c = LSTM(embed(x), h, c)
    x = argmax(fc(h))
```

In [ ]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_size):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.lstm  = nn.LSTM(embed_dim, hidden_size, batch_first=True)

    def forward(self, src):
        """src: (B, T) → context: ((1,B,H), (1,B,H))"""
        emb = self.embed(src)          # (B, T, embed_dim)
        _, (h, c) = self.lstm(emb)    # h,c: (1, B, H)
        return h, c                   # the bottleneck


class Decoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_size):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.lstm  = nn.LSTM(embed_dim, hidden_size, batch_first=True)
        self.fc    = nn.Linear(hidden_size, vocab_size)

    def forward(self, tgt_in, h, c):
        """Teacher-forced forward: tgt_in (B,T) → logits (B,T,vocab)."""
        emb = self.embed(tgt_in)       # (B, T, embed_dim)
        out, _ = self.lstm(emb, (h,c)) # (B, T, H)
        return self.fc(out)            # (B, T, vocab)

    def decode_greedy(self, h, c, max_len=20):
        """Autoregressive decoding: generate until EOS or max_len."""
        B = h.size(1)
        x = torch.full((B, 1), SOS, dtype=torch.long, device=h.device)
        outputs = []
        for _ in range(max_len):
            emb = self.embed(x)        # (B, 1, embed_dim)
            out, (h, c) = self.lstm(emb, (h, c))
            token = self.fc(out).argmax(-1)  # (B, 1)
            outputs.append(token)
            x = token
        return torch.cat(outputs, dim=1)  # (B, max_len)


EMBED_DIM   = 32
HIDDEN_SIZE = 128
encoder = Encoder(VOCAB_SIZE, EMBED_DIM, HIDDEN_SIZE).to(device)
decoder = Decoder(VOCAB_SIZE, EMBED_DIM, HIDDEN_SIZE).to(device)

params = sum(p.numel() for p in list(encoder.parameters()) + list(decoder.parameters()))
print(f"Total parameters: {params:,}")

In [ ]:
# Training on sequences of length 5
TRAIN_SEQ_LEN = 5
BATCH_SIZE = 64
N_STEPS = 2000

optimizer = torch.optim.Adam(
    list(encoder.parameters()) + list(decoder.parameters()), lr=1e-3)
criterion = nn.CrossEntropyLoss(ignore_index=PAD)

losses = []
for step in range(N_STEPS):
    src, tgt_in, tgt_out = make_batch(BATCH_SIZE, TRAIN_SEQ_LEN)
    src     = src.to(device)
    tgt_in  = tgt_in.to(device)
    tgt_out = tgt_out.to(device)

    h, c = encoder(src)
    logits = decoder(tgt_in, h, c)   # (B, T, vocab)
    loss = criterion(logits.reshape(-1, VOCAB_SIZE), tgt_out.reshape(-1))

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(
        list(encoder.parameters()) + list(decoder.parameters()), 1.0)
    optimizer.step()
    losses.append(loss.item())

    if (step + 1) % 500 == 0:
        print(f"Step {step+1} | loss: {loss.item():.4f}")

In [ ]:
def evaluate_accuracy(seq_len, n_samples=500):
    """Exact sequence accuracy for greedy decoding on sequences of given length."""
    encoder.eval(); decoder.eval()
    correct = 0
    with torch.no_grad():
        src, _, tgt_out = make_batch(n_samples, seq_len)
        src = src.to(device)
        h, c = encoder(src)
        pred = decoder.decode_greedy(h, c, max_len=seq_len+1)  # (B, seq_len+1)
        pred_tokens = pred[:, :seq_len].cpu()
        true_tokens = tgt_out[:, :seq_len]
        correct = (pred_tokens == true_tokens).all(dim=1).sum().item()
    return correct / n_samples

# Test accuracy at different sequence lengths — trained only on length 5!
test_lengths = [3, 5, 7, 10, 15, 20]
accuracies = [evaluate_accuracy(L) for L in test_lengths]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

window = 50
smooth = np.convolve(losses, np.ones(window)/window, 'valid')
ax1.plot(smooth, 'steelblue', lw=2)
ax1.set_xlabel('Training step'); ax1.set_ylabel('Loss')
ax1.set_title('Seq2Seq training loss (reversal, length=5)')
ax1.grid(True, alpha=0.3)

ax2.bar(range(len(test_lengths)), accuracies, color='steelblue', alpha=0.8)
ax2.set_xticks(range(len(test_lengths)))
ax2.set_xticklabels([f'len={L}' for L in test_lengths])
ax2.set_ylabel('Exact sequence accuracy')
ax2.set_title('THE BOTTLENECK: accuracy collapses with longer sequences')
ax2.set_ylim(0, 1.05)
ax2.axvline(1.5, color='tomato', linestyle='--', lw=1.5, label='train length')
ax2.legend(); ax2.grid(True, alpha=0.3, axis='y')

for i, (L, acc) in enumerate(zip(test_lengths, accuracies)):
    ax2.text(i, acc + 0.02, f'{acc:.2f}', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# Show some examples
encoder.eval(); decoder.eval()
print("=== Decoding Examples ===")
for test_len in [5, 10, 15]:
    src, _, tgt_out = make_batch(5, test_len)
    src_d = src.to(device)
    with torch.no_grad():
        h, c = encoder(src_d)
        pred = decoder.decode_greedy(h, c, max_len=test_len+1)
    print(f"\nLength {test_len}:")
    for i in range(min(3, 5)):
        s = src[i].tolist()
        t = tgt_out[i, :test_len].tolist()
        p = pred[i, :test_len].cpu().tolist()
        match = '✓' if p == t else '✗'
        print(f"  {match} Input: {s} | Expected: {t} | Got: {p}")

## The Bottleneck Visualized

The encoder must compress an entire input sequence into **one fixed-size vector** (h, c).
For short sequences, this vector has enough capacity. For longer sequences,
information is inevitably lost — especially elements from the beginning of the sequence
that are far from the final hidden state.

The decoder receives **the same context vector** regardless of which output position it's generating.
It must simultaneously remember the entire input to generate all outputs.

**The fix**: instead of a single context vector, give the decoder access to **all encoder hidden states**,
and let it pick which ones to attend to at each step. That's **attention** — next notebook.

In [ ]:
# Visualize the context vector — how much information fits?
# Compare the norm of (h,c) for different input lengths
encoder.eval()
norms = []
lengths_for_norm = range(2, 25)
with torch.no_grad():
    for L in lengths_for_norm:
        src, _, _ = make_batch(200, L)
        h, c = encoder(src.to(device))
        combined = torch.cat([h.squeeze(0), c.squeeze(0)], dim=1)  # (B, 2H)
        norms.append(combined.norm(dim=1).mean().item())

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(list(lengths_for_norm), norms, 'steelblue', lw=2, marker='o', markersize=5)
ax.set_xlabel('Input sequence length')
ax.set_ylabel('Mean context vector norm ||[h,c]||')
ax.set_title('Context vector norm saturates as input grows longer — information is compressed')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Key Takeaways

| Concept | What we learned |
|---|---|
| **Encoder** | Reads input, produces fixed-size context vector (h, c) |
| **Decoder** | Initialized from context; generates output autoregressively |
| **Teacher forcing** | Feed ground-truth targets during training; faster convergence |
| **Bottleneck problem** | One vector must encode the entire input — fails for long sequences |
| **Exposure bias** | Train on ground truth; test on own predictions — error accumulates |
| **Greedy decoding** | Good approximation; beam search gives better results in practice |

**The key insight from this notebook**: performance degrades predictably as sequence length grows.
A single fixed-size vector cannot represent arbitrarily long sequences without information loss.

**The fix**: give the decoder access to **all** encoder hidden states and compute a **per-step
context vector** as a weighted sum. That's Bahdanau attention — next notebook.